In [ ]:
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
import pandas as pd

# for the frequency spectra
import multitaper as mt
from scipy import stats

def multitaper_spectrum(x, dt, nw=2):
    """Multitaper spectrum + classical chi-squared CI + Thomson F-test significance mask."""
    kspec = int(2*nw - 1)   # number of tapers - more tapers = less variance, less resolution
    psd = mt.MTSpec(x, nw=nw, kspec=kspec, dt=dt, iadapt=0)

    freq, spec = psd.rspec()
    freq = freq.squeeze()
    spec = spec.squeeze()

    one_sided = psd.freq.squeeze() >= 0

    # classical chi-squared CI (jackspec() proved unreliable - see SR-LPNE_DWT.ipynb)
    dof = 2 * kspec
    ci_lower = spec * dof / stats.chi2.ppf(0.975, dof)
    ci_upper = spec * dof / stats.chi2.ppf(0.025, dof)

    F, _ = psd.ftest()
    F = F.squeeze()[one_sided]
    f_crit = stats.f.ppf(0.95, 2, 2*kspec - 2)   # df1=2, df2=2K-2 for the harmonic F-test
    significant = F > f_crit

    return freq, spec, ci_lower, ci_upper, significant

# for cross correlation
from scipy import signal

def cross_correlate(x, y, dt, max_lag):
    """
    Normalized cross-correlation of x and y, and the lag (in the same units as dt)
    that maximizes it, restricted to |lag| <= max_lag.

    Sign convention (scipy.signal.correlate(x, y)): a NEGATIVE best lag means y
    lags behind x (y's pattern happens later); a POSITIVE best lag means y leads x.
    """
    x = x - x.mean()
    y = y - y.mean()
    corr = signal.correlate(x, y, mode="full")
    corr = corr / (len(x) * x.std() * y.std())  # normalize so a perfect match peaks near 1.0
    lags = signal.correlation_lags(len(x), len(y), mode="full") * dt

    mask = np.abs(lags) <= max_lag
    lags, corr = lags[mask], corr[mask]
    best_lag = lags[np.argmax(corr)]
    worst_lag = lags[np.argmin(corr)]
    return lags, corr, best_lag, worst_lag

def describe_lag(label, best_lag):
    if best_lag < 0:
        print(f"{label}: model LAGS ground truth by {abs(best_lag):.2f} kyr")
    elif best_lag > 0:
        print(f"{label}: model LEADS ground truth by {best_lag:.2f} kyr")
    else:
        print(f"{label}: no lag (perfectly in phase)")

In [ ]:
SR_model = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/ModelOutput/SR_test_rollout.csv")
hMSER_model = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/ModelOutput/hMSE_test_rollout.csv")
gt = pd.read_csv("/Users/vivekveluvali/Desktop/KU_Classes/MPT_Emulator_Project/LPNE/Data/InterpolatedObs/gpr_full_traj.csv")

In [ ]:
SR_model.head()

In [ ]:
hMSER_model.head()

In [ ]:
gt.head()

In [ ]:
fig,ax = plt.subplots(figsize=(20,10),sharex=True)
    
ax.plot(gt["Age[kaBP]"], gt["dO18_LR04[pm]"], label="Ground Truth", color="black")
ax.plot(SR_model["Age[kaBP]"], SR_model["dO18_LR04"], label="SR Model", color="#D6195E", alpha = 0.7)
ax.plot(hMSER_model["Age[kaBP]"], hMSER_model["dO18_LR04"], label="hMSE Model", color="#43A047", alpha = 0.7)
ax.set_xlabel("Age [kaBP]")
ax.set_ylabel("dO18_LR04[pm]")
ax.set_title("Comparison of dO18_LR04 Predictions")
ax.legend()

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(20,10),sharex=True)
axes[0].plot(hMSER_model["Age[kaBP]"][0:43], hMSER_model["dO18_LR04"][0:43], label="dO18", color="black")
axes[1].plot(gt["Age[kaBP]"][1200:1243], gt["Insolation[W/m2]"][1200:1243], label="Insolation", color="black")

axes[0].vlines(x = 1200, ymin = 3, ymax = 5)
axes[0].vlines(x = 1200 + 13*1, ymin = 3, ymax = 5)
axes[0].vlines(x = 1200 + 13*2, ymin = 3, ymax = 5)
axes[0].vlines(x = 1200 + 13*3, ymin = 3, ymax = 5)
axes[0].vlines(x = 1200 + 13*4, ymin = 3, ymax = 5)

axes[1].vlines(x = 1200, ymin = 6, ymax = 7)
axes[1].vlines(x = 1200 + 13*1, ymin = 6, ymax = 7)
axes[1].vlines(x = 1200 + 13*2, ymin = 6, ymax = 7)
axes[1].vlines(x = 1200 + 13*3, ymin = 6, ymax = 7)
axes[1].vlines(x = 1200 + 13*4, ymin = 6, ymax = 7)

In [ ]:
fig, axes = plt.subplots(2,1,figsize=(20,10),sharex=True)
axes[0].plot(hMSER_model["Age[kaBP]"], hMSER_model["dO18_LR04"], label="dO18", color="black")
axes[1].plot(gt["Age[kaBP]"], gt["Insolation[W/m2]"], label="Insolation", color="black")


In [ ]:
dt = 1.0  # kyr - all three series are on the same 1 ka BP grid (0-1243 kaBP, 1244 rows)
nw = 2

# label -> (series, color)
datasets = {
    "gt":   ("Ground truth", gt["dO18_LR04[pm]"].to_numpy(),      "#000000"),
    "sr":   ("SR model",     SR_model["dO18_LR04"].to_numpy(),    "#D6195E"),
    "hmse": ("hMSE model",   hMSER_model["dO18_LR04"].to_numpy(), "#8DB1A7"),
}

results = {}
fig, ax = plt.subplots(figsize=(12, 6))

for key, (label, x, color) in datasets.items():
    freq, spec, ci_lower, ci_upper, significant = multitaper_spectrum(x, dt, nw)
    results[key] = {"freq": freq, "spec": spec, "ci_lower": ci_lower, "ci_upper": ci_upper, "significant": significant}

    ax.plot(freq, spec, color=color, label=f"{label} spectrum")
    ax.fill_between(freq, ci_lower, ci_upper, color=color, alpha=0.15)
    ax.scatter(freq[significant], spec[significant], color=color, edgecolor="black", s=20, zorder=5)

    print(f"{label} significant periods (kyr):", 1 / freq[significant])

ymin = min(r["spec"].min() for r in results.values())
ymax = max(r["spec"].max() for r in results.values())
ax.vlines(1/100, ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", label="100 kyr cycle", color="slategrey")
ax.vlines(1/41,  ymin=ymin, ymax=ymax, alpha=0.5, linestyle="dashed", label="41 kyr cycle",  color="darkblue")

ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("frequency [1/kyr]")
ax.set_ylabel("power")
ax.set_title("Multitaper spectra: dO18_LR04 - ground truth vs. SR vs. hMSE model rollouts")
ax.legend()
plt.tight_layout()
plt.show()

# unpacked, suffixed variables kept for any later cells that reference them directly
freq_gt, spec_gt, ci_lower_gt, ci_upper_gt, significant_gt = (results["gt"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))
freq_sr, spec_sr, ci_lower_sr, ci_upper_sr, significant_sr = (results["sr"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))
freq_hmse, spec_hmse, ci_lower_hmse, ci_upper_hmse, significant_hmse = (results["hmse"][k] for k in ("freq", "spec", "ci_lower", "ci_upper", "significant"))

In [ ]:
# --- lowpass filter (Butterworth) at ~0.1 (1/kyr), applied zero-phase via filtfilt ---
dt = 1.0        # kyr, sampling interval (all three series share this grid)
fs = 1.0 / dt   # sampling frequency, 1/kyr
cutoff = 0.1    # 1/kyr - passes periods longer than ~10 kyr
order = 4

b, a = signal.butter(order, cutoff, btype="low", fs=fs)

# filtfilt (not lfilter) specifically because it's zero-phase - lfilter would
# introduce its own phase shift, which would corrupt the lag we're about to measure
gt_filt   = signal.filtfilt(b, a, gt["dO18_LR04[pm]"].to_numpy())
sr_filt   = signal.filtfilt(b, a, SR_model["dO18_LR04"].to_numpy())
hmse_filt = signal.filtfilt(b, a, hMSER_model["dO18_LR04"].to_numpy())
gt_i_filt   = signal.filtfilt(b, a, gt["Insolation[W/m2]"].to_numpy())
sr_i_filt   = signal.filtfilt(b, a, SR_model["Insolation[W/m2]"].to_numpy())
hmse_i_filt = signal.filtfilt(b, a, hMSER_model["Insolation[W/m2]"].to_numpy())

# --- cross-correlation: which lag maximizes correlation between gt and each model? ---
max_lag_kyr = 100  # restrict the search to physically sensible lags - at the far
                    # edges of a "full" cross-correlation the overlap between the
                    # two series shrinks to just a few points, which can produce
                    # noisy/spurious peaks despite the normalization below

lags_gt, corr_gt, best_lag_gt, worst_lag_gt = cross_correlate(gt_filt, gt_i_filt, dt, max_lag_kyr)
lags_hmse, corr_hmse, best_lag_hmse, worst_lag_hmse = cross_correlate(hmse_filt, hmse_i_filt, dt, max_lag_kyr)
lags_sr, corr_sr, best_lag_sr, worst_lag_sr = cross_correlate(sr_filt, sr_i_filt, dt, max_lag_kyr)


describe_lag("SR model", best_lag_sr)
describe_lag("hMSE model", best_lag_hmse)
describe_lag("ground truth", best_lag_gt)

# --- plot ---
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axs[0].plot(lags_sr, corr_sr, color="#D6195E", label="SR model")
axs[0].axvline(best_lag_sr, color="#D6195E", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_sr:.2f} kyr")
#axs[0].axvline(worst_lag_sr, color="#D6195E", linestyle="--", alpha=0.7,
#               label=f"worst lag = {worst_lag_sr:.2f} kyr")
axs[0].set_ylabel("normalized cross-correlation")
axs[0].set_title("Cross-correlation: ground truth vs. SR model (lowpass filtered, dO18_LR04)")
axs[0].legend()
axs[0].grid(alpha=0.3)

axs[1].plot(lags_hmse, corr_hmse, color="#79C7B2", label="DWT-hMSE model")
axs[1].axvline(best_lag_hmse, color="#79C7B2", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_hmse:.2f} kyr")
axs[1].set_ylabel("normalized cross-correlation")
axs[1].legend()
axs[1].grid(alpha=0.3)

axs[2].plot(lags_gt, corr_gt, color="black", label="ground truth")
axs[2].axvline(best_lag_gt, color="black", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_gt:.2f} kyr")
axs[2].set_ylabel("normalized cross-correlation")
axs[2].set_xlabel("lag [kyr]")
axs[2].set_title("Cross-correlation: ground truth vs. ground truth (lowpass filtered, dO18_LR04)")
axs[2].legend()
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# --- bandpass filter (Butterworth) isolating the ~41 kyr obliquity cycle:
# passband = periods 38-43 kyr, i.e. frequencies 1/43-1/38 (1/kyr) ---
dt = 1.0
fs = 1.0 / dt
period_low, period_high = 40, 42     # kyr
freq_low, freq_high = 1.0/period_high, 1.0/period_low   # 1/kyr
order = 4

b_b41, a_b41 = signal.butter(order, [freq_low, freq_high], btype="band", fs=fs)

gt_b41   = signal.filtfilt(b_b41, a_b41, gt["dO18_LR04[pm]"].to_numpy())
sr_b41   = signal.filtfilt(b_b41, a_b41, SR_model["dO18_LR04"].to_numpy())
hmse_b41 = signal.filtfilt(b_b41, a_b41, hMSER_model["dO18_LR04"].to_numpy())
gt_i_b41   = signal.filtfilt(b_b41, a_b41, gt["Insolation[W/m2]"].to_numpy())
sr_i_b41   = signal.filtfilt(b_b41, a_b41, SR_model["Insolation[W/m2]"].to_numpy())
hmse_i_b41 = signal.filtfilt(b_b41, a_b41, hMSER_model["Insolation[W/m2]"].to_numpy())

# --- cross-correlation within this band: which lag maximizes correlation between
# the band-limited state variable and the band-limited insolation forcing? ---
max_lag_kyr = 20.5  # +/- half the isolated cycle - see comment above

lags_gt_b41, corr_gt_b41, best_lag_gt_b41, worst_lag_gt_b41 = cross_correlate(gt_b41, gt_i_b41, dt, max_lag_kyr)
lags_hmse_b41, corr_hmse_b41, best_lag_hmse_b41, worst_lag_hmse_b41 = cross_correlate(hmse_b41, hmse_i_b41, dt, max_lag_kyr)
lags_sr_b41, corr_sr_b41, best_lag_sr_b41, worst_lag_sr_b41 = cross_correlate(sr_b41, sr_i_b41, dt, max_lag_kyr)

describe_lag("SR model (41 kyr band)", best_lag_sr_b41)
describe_lag("hMSE model (41 kyr band)", best_lag_hmse_b41)
describe_lag("ground truth (41 kyr band)", best_lag_gt_b41)

# --- plot ---
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axs[0].plot(lags_sr_b41, corr_sr_b41, color="#D6195E", label="SR model")
axs[0].axvline(best_lag_sr_b41, color="#D6195E", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_sr_b41:.2f} kyr")
axs[0].set_ylabel("normalized cross-correlation")
axs[0].set_title("Cross-correlation (40-42 kyr band, isolating ~41 kyr obliquity cycle): model dO18 vs. model insolation")
axs[0].legend()
axs[0].grid(alpha=0.3)

axs[1].plot(lags_hmse_b41, corr_hmse_b41, color="#79C7B2", label="DWT-hMSE model")
axs[1].axvline(best_lag_hmse_b41, color="#79C7B2", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_hmse_b41:.2f} kyr")
axs[1].set_ylabel("normalized cross-correlation")
axs[1].legend()
axs[1].grid(alpha=0.3)

axs[2].plot(lags_gt_b41, corr_gt_b41, color="black", label="ground truth")
axs[2].axvline(best_lag_gt_b41, color="black", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_gt_b41:.2f} kyr")
axs[2].set_ylabel("normalized cross-correlation")
axs[2].set_xlabel("lag [kyr]")
axs[2].set_title("Cross-correlation (40-42 kyr band): ground truth dO18 vs. ground truth insolation")
axs[2].legend()
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# --- bandpass filter (Butterworth) isolating the ~100 kyr eccentricity cycle:
# passband = periods 90-110 kyr, i.e. frequencies 1/110-1/90 (1/kyr) ---
dt = 1.0
fs = 1.0 / dt
period_low, period_high = 90, 110     # kyr
freq_low, freq_high = 1.0/period_high, 1.0/period_low   # 1/kyr
order = 4

b_b100, a_b100 = signal.butter(order, [freq_low, freq_high], btype="band", fs=fs)

gt_b100   = signal.filtfilt(b_b100, a_b100, gt["dO18_LR04[pm]"].to_numpy())
sr_b100   = signal.filtfilt(b_b100, a_b100, SR_model["dO18_LR04"].to_numpy())
hmse_b100 = signal.filtfilt(b_b100, a_b100, hMSER_model["dO18_LR04"].to_numpy())
gt_i_b100   = signal.filtfilt(b_b100, a_b100, gt["Insolation[W/m2]"].to_numpy())
sr_i_b100   = signal.filtfilt(b_b100, a_b100, SR_model["Insolation[W/m2]"].to_numpy())
hmse_i_b100 = signal.filtfilt(b_b100, a_b100, hMSER_model["Insolation[W/m2]"].to_numpy())

# --- cross-correlation within this band: which lag maximizes correlation between
# the band-limited state variable and the band-limited insolation forcing? ---
max_lag_kyr = 50.0  # +/- half the isolated cycle - see comment above

lags_gt_b100, corr_gt_b100, best_lag_gt_b100, worst_lag_gt_b100 = cross_correlate(gt_b100, gt_i_b100, dt, max_lag_kyr)
lags_hmse_b100, corr_hmse_b100, best_lag_hmse_b100, worst_lag_hmse_b100 = cross_correlate(hmse_b100, hmse_i_b100, dt, max_lag_kyr)
lags_sr_b100, corr_sr_b100, best_lag_sr_b100, worst_lag_sr_b100 = cross_correlate(sr_b100, sr_i_b100, dt, max_lag_kyr)

describe_lag("SR model (100 kyr band)", best_lag_sr_b100)
describe_lag("hMSE model (100 kyr band)", best_lag_hmse_b100)
describe_lag("ground truth (100 kyr band)", best_lag_gt_b100)

# --- plot ---
fig, axs = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axs[0].plot(lags_sr_b100, corr_sr_b100, color="#D6195E", label="SR model")
axs[0].axvline(best_lag_sr_b100, color="#D6195E", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_sr_b100:.2f} kyr")
axs[0].set_ylabel("normalized cross-correlation")
axs[0].set_title("Cross-correlation (90-110 kyr band, isolating ~100 kyr eccentricity cycle): ground truth vs. SR model")
axs[0].legend()
axs[0].grid(alpha=0.3)

axs[1].plot(lags_hmse_b100, corr_hmse_b100, color="#79C7B2", label="DWT-hMSE model")
axs[1].axvline(best_lag_hmse_b100, color="#79C7B2", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_hmse_b100:.2f} kyr")
axs[1].set_ylabel("normalized cross-correlation")
axs[1].legend()
axs[1].grid(alpha=0.3)

axs[2].plot(lags_gt_b100, corr_gt_b100, color="black", label="ground truth")
axs[2].axvline(best_lag_gt_b100, color="black", linestyle="--", alpha=0.7,
               label=f"peak lag = {best_lag_gt_b100:.2f} kyr")
axs[2].set_ylabel("normalized cross-correlation")
axs[2].set_xlabel("lag [kyr]")
axs[2].set_title("Cross-correlation (90-110 kyr band): ground truth vs. ground truth")
axs[2].legend()
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()
